In [1]:
import pandas as pd
import pdfplumber
import re
from typing import Dict, Tuple, Any
import os
from decimal import Decimal
from tqdm import tqdm

In [2]:
table_name = {
    "core_performance_indicators_sheet":[
        "主要会计数据",
        "主要财务指标",
    ],
    "balance_sheet":[
        "合并资产负债表",
        "合并年初到报告期末资产负债表"
    ],
    "cash_flow_sheet":[
        "合并现金流量表",
        "合并年初到报告期末现金流量表",
    ],
    "income_sheet":[
        "合并利润表",
        "合并年初到报告期末利润表",
    ],
    "other":[
        "主要会计数据",
        "主要财务指标",
        "合并资产负债表",
        "合并年初到报告期末资产负债表",
    ]
}

key_words = {
    "core_performance_indicators_sheet":{
        "eps":["基本每股收益"],
        "total_operating_revenue":["营业总收入","营业收入"],
        "net_profit_10k_yuan":["归属于上市公司股东的净利润"],
        "roe":["加权平均净资产收益率"],
        "net_profit_excl_non_recurring":["归属于上市公司股东的扣除非经常性损益的净利润"],
        "roe_weighted_excl_non_recurring":["扣除非经常性损益后的加权平均净资产收益率"]
    },
    "balance_sheet":{
        "asset_cash_and_cash_equivalents":["货币资金"],
        "asset_accounts_receivable":["应收账款"],
        "asset_inventory":["存货"],
        "asset_trading_financial_assets":["交易性金融资产"],
        "asset_construction_in_progress":["在建工程"],
        "asset_total_assets":["资产总计","总资产"],
        "liability_accounts_payable":["应付账款"],
        "liability_advance_from_customers":["预收款项","预收账款"],
        "liability_total_liabilities":["负债合计","总负债"],
        "liability_contract_liabilities":["合同负债"],
        "liability_short_term_loans":["短期借款"],
        "equity_unappropriated_profit":["未分配利润"],
        "equity_total_equity":["所有者权益（或股东权益）合计", "所有者权益合计"],
    },
    "cash_flow_sheet":{
        "net_cash_flow":["现金及现金等价物净增加额"],
        "operating_cf_net_amount":["经营活动产生的现金流量净额"],
        "operating_cf_cash_from_sales":["销售商品、提供劳务收到的现金"],
        "investing_cf_net_amount":["投资活动产生的现金流量净额"],
        "investing_cf_cash_for_investments":["投资支付的现金"],
        "investing_cf_cash_from_investment_recovery":["收回投资收到的现金"],
        "financing_cf_cash_from_borrowing":["取得借款收到的现金"],
        "financing_cf_cash_for_debt_repayment":["偿还债务支付的现金"],
        "financing_cf_net_amount":["筹资活动产生的现金流量净额"],
    },
    "income_sheet":{
        "net_profit":["净利润（净亏损以“－”号填列）", "净利润（净亏损以“-”号填列）", "净利润（净亏损以"],
        "other_income":["其他收益"],
        "total_operating_revenue":["营业总收入"],
        "operating_expense_cost_of_sales":["营业成本"],
        "operating_expense_selling_expenses":["销售费用"],
        "operating_expense_administrative_expenses":["管理费用"],
        "operating_expense_financial_expenses":["财务费用"],
        "operating_expense_rnd_expenses":["研发费用"],
        "operating_expense_taxes_and_surcharges":["税金及附加"],
        "total_operating_expenses":["营业总支出","营业总成本"],
        "operating_profit":["营业利润"],
        "total_profit":["利润总额"],
        "asset_impairment_loss":["资产减值损失"],
        "credit_impairment_loss":["信用减值损失"],
    },
    "other":{
        "total_guben_value":["实收资本（或股本）", "股本", "实收资本"],
        "total_jingzichan":["归属于上市公司股东的净资产", "归属于上市公司股东的所有者权益", "归属于母公司所有者权益合计"],
    }
}

skip_words = {
    "发生变动的情况及原因",
    "变动原因",
    "变动比率"
}

In [3]:
table_columns = {
    "core_performance_indicators_sheet": [
        "serial_number",
        "stock_code",
        "stock_abbr",
        "report_period",
        "report_year",
        "eps",
        "total_operating_revenue",
        "operating_revenue_yoy_growth",
        "operating_revenue_qoq_growth",
        "net_profit_10k_yuan",
        "net_profit_yoy_growth",
        "net_profit_qoq_growth",
        "net_asset_per_share",
        "roe",
        "operating_cf_per_share",
        "net_profit_excl_non_recurring",
        "net_profit_excl_non_recurring_yoy",
        "gross_profit_margin",
        "net_profit_margin",
        "roe_weighted_excl_non_recurring",
    ],
    "balance_sheet": [
        "serial_number",
        "stock_code",
        "stock_abbr",
        "report_period",
        "report_year",
        "asset_cash_and_cash_equivalents",
        "asset_accounts_receivable",
        "asset_inventory",
        "asset_trading_financial_assets",
        "asset_construction_in_progress",
        "asset_total_assets",
        "asset_total_assets_yoy_growth",
        "liability_accounts_payable",
        "liability_advance_from_customers",
        "liability_total_liabilities",
        "liability_total_liabilities_yoy_growth",
        "liability_contract_liabilities",
        "liability_short_term_loans",
        "asset_liability_ratio",
        "equity_unappropriated_profit",
        "equity_total_equity",
    ],
    "cash_flow_sheet": [
        "serial_number",
        "stock_code",
        "stock_abbr",
        "report_period",
        "report_year",
        "net_cash_flow",
        "net_cash_flow_yoy_growth",
        "operating_cf_net_amount",
        "operating_cf_ratio_of_net_cf",
        "operating_cf_cash_from_sales",
        "investing_cf_net_amount",
        "investing_cf_ratio_of_net_cf",
        "investing_cf_cash_for_investments",
        "investing_cf_cash_from_investment_recovery",
        "financing_cf_cash_from_borrowing",
        "financing_cf_cash_for_debt_repayment",
        "financing_cf_net_amount",
        "financing_cf_ratio_of_net_cf",
    ],
    "income_sheet": [
        "serial_number",
        "stock_code",
        "stock_abbr",
        "report_period",
        "report_year",
        "net_profit",
        "net_profit_yoy_growth",
        "other_income",
        "total_operating_revenue",
        "operating_revenue_yoy_growth",
        "operating_expense_cost_of_sales",
        "operating_expense_selling_expenses",
        "operating_expense_administrative_expenses",
        "operating_expense_financial_expenses",
        "operating_expense_rnd_expenses",
        "operating_expense_taxes_and_surcharges",
        "total_operating_expenses",
        "operating_profit",
        "total_profit",
        "asset_impairment_loss",
        "credit_impairment_loss",
    ],
    "other": [
        "total_guben_value",
        "total_jingzichan",
    ]
}

In [4]:
def is_change_column(col_name: str) -> bool:
    s = normalize(col_name)
    return any(k in s for k in ["增减", "同比", "比上年同期", "变动"])


def reorder_period_columns(period_cols, all_cols):
    """
    规则：
    1. 先显示数值列
    2. 再显示同比/增减列
    3. 同类之间保持表格原始从左到右顺序
    """
    pos_map = {str(col): i for i, col in enumerate(all_cols)}

    return sorted(
        [str(col) for col in period_cols],
        key=lambda col: (
            1 if is_change_column(col) else 0,
            pos_map.get(str(col), 10**9)
        )
    )

In [5]:
# def parse_numeric_value(x):
#     if x is None:
#         return None

#     if isinstance(x, pd.Series):
#         x = x.dropna()
#         if len(x) == 0:
#             return None
#         x = x.iloc[0]

#     s = str(x).strip()
#     if s == "":
#         return None

#     if re.fullmatch(r"[-+]?[\d,]+(\.\d+)?", s):
#         return float(s.replace(",", ""))

#     if re.fullmatch(r"[-+]?[\d,]+(\.\d+)?%", s):
#         return s.replace(",", "")

#     return None

def parse_numeric_value(x):
    if x is None:
        return None

    if isinstance(x, pd.Series):
        for item in x.tolist():
            val = parse_numeric_value(item)
            if val is not None:
                return val
        return None

    s = str(x).strip()
    if s == "":
        return None

    s = s.replace("，", ",")

    if re.fullmatch(r"[-+]?[\d,]+(\.\d+)?", s):
        return float(s.replace(",", ""))

    if re.fullmatch(r"[-+]?[\d,]+(\.\d+)?%", s):
        return s.replace(",", "")

    return None

In [6]:
def get_year_key(col_name: str) -> str:
    s = normalize(col_name)
    m = re.search(r"(20\d{2})", s)
    if m:
        return m.group(1)
    return s


def get_adjust_priority(col_name: str) -> int:
    s = normalize(col_name)
    if "调整后" in s or "重述后" in s:
        return 3
    if "调整前" in s or "重述前" in s:
        return 1
    return 2

In [7]:
def is_annual_report(pdf_path: str = "") -> bool:
    s = normalize(pdf_path)
    annual_keywords = ["年度报告", "年报"]
    quarterly_keywords = ["一季度报告", "第一季度报告", "半年报", "半年度报告", "三季度报告", "第三季度报告"]
    if any(k in s for k in annual_keywords) and not any(k in s for k in quarterly_keywords):
        return True
    return False


def is_quarter_column(col_name: str) -> bool:
    s = normalize(col_name)
    quarter_patterns = [
        "第一季度", "第二季度", "第三季度", "第四季度",
        "1-3月", "1-6月", "1-9月",
        "3月31日", "6月30日", "9月30日"
    ]
    return any(p in s for p in quarter_patterns)


def is_annual_column(col_name: str, table_name: str = "") -> bool:
    s = normalize(col_name)
    table_name = normalize(table_name)

    # 年度类列
    annual_patterns = [
        "年度", "年末", "12月31日", "本报告期末", "上年末", "上年度末",
        "年初至报告期末", "本年累计", "本报告期累计",
        "上年初至上年报告期末", "上年同期累计",
        "期末余额", "期初余额",
        "调整前", "调整后",
    ]
    if any(p in s for p in annual_patterns):
        return True

    # 包含年份通常也是年报优先列
    if re.search(r"20\d{2}", s):
        return True

    return False

In [8]:
def normalize(s):
    if s is None:
        return ""

    s = str(s).replace("\u3000", " ")
    s = s.replace("（", "(").replace("）", ")")
    s = s.replace("％", "%")
    s = re.sub(r"\s+", "", s)

    s = s.strip()
    return s


def clean_cell(x):
    if x is None:
        return ""
    return str(x).strip().replace("\n", "").replace(" ", "")


def normalize_rows(rows):
    if not rows:
        return []

    max_len = max(len(r) for r in rows)
    result = []
    for r in rows:
        r = list(r)
        if len(r) < max_len:
            r = r + [None] * (max_len - len(r))
        result.append(r)
    return result


def is_numeric_like(x):
    s = clean_cell(x)
    if s == "":
        return False

    s2 = s.replace(",", "").replace("%", "").replace("，", "")
#     s2 = s.replace(",", "").replace("，", "")
    try:
        float(s2)
        return True
    except:
        return False


def is_footer_label(x):
    s = clean_cell(x)
    keywords = ["年末", "期末", "本年末", "上年末", "本期末", "上期期末"]
    return any(k in s for k in keywords)


def drop_empty_columns(header, data_rows):
    if not header:
        return header, data_rows

    header = [clean_cell(h) for h in header]
    data_rows = normalize_rows(data_rows)

    keep_idx = []
    for j, h in enumerate(header):
        col_values = [clean_cell(row[j]) if j < len(row) else "" for row in data_rows]
        real_values = [v for v in col_values if v != "" and not is_footer_label(v)]
        has_data = any(v != "" for v in real_values)

        keep = True
        if h == "" and not has_data:
            keep = False

        if not has_data:
            left_same = (j > 0 and header[j - 1] == h and h != "")
            right_same = (j < len(header) - 1 and header[j + 1] == h and h != "")
            if left_same or right_same:
                keep = False

        if keep:
            keep_idx.append(j)

    new_header = [header[j] for j in keep_idx]
    new_rows = [[row[j] if j < len(row) else "" for j in keep_idx] for row in data_rows]
    return new_header, new_rows


def find_special_column_index(header):
    priority_keywords = ["调整前", "调整后", "重述前", "重述后"]
    cleaned_header = [clean_cell(h) for h in header]

    for kw in priority_keywords:
        for i, h in enumerate(cleaned_header):
            if kw in h:
                return i
    return -1


def fix_misaligned_year_columns(header, data_rows):
    if not header:
        return header, data_rows

    header = [clean_cell(h) for h in header]
    data_rows = normalize_rows(data_rows)
    special_keywords = ["调整前", "调整后", "重述前", "重述后"]
    to_delete = set()

    for i, h in enumerate(header):
        if any(k in h for k in special_keywords):
            base_year = h.split("_")[0] if "_" in h else h

            left = i - 1
            if left >= 0 and header[left] == base_year:
                for row in data_rows:
                    if i < len(row) and left < len(row):
                        left_val = clean_cell(row[left])
                        cur_val = clean_cell(row[i])
                        if left_val != "" and cur_val == "":
                            row[i] = row[left]
                            row[left] = ""
                col_vals = [clean_cell(r[left]) if left < len(r) else "" for r in data_rows]
                if all(v == "" for v in col_vals):
                    to_delete.add(left)

            right = i + 1
            if right < len(header) and header[right] == base_year:
                col_vals = [clean_cell(r[right]) if right < len(r) else "" for r in data_rows]
                if all(v == "" for v in col_vals):
                    to_delete.add(right)

    keep_idx = [j for j in range(len(header)) if j not in to_delete]
    new_header = [header[j] for j in keep_idx]
    new_rows = [[row[j] if j < len(row) else "" for j in keep_idx] for row in data_rows]
    return new_header, new_rows


def fix_shifted_columns(header, data_rows):
    if not header:
        return header, data_rows

    header = [clean_cell(h) for h in header]
    data_rows = normalize_rows(data_rows)
    n = len(header)
    to_delete = set()

    for j in range(1, n):
        cur_h = header[j]
        left_h = header[j - 1]
        col_vals = [clean_cell(row[j]) if j < len(row) else "" for row in data_rows]
        non_empty_vals = [v for v in col_vals if v != ""]

        if cur_h != "" and cur_h == left_h and non_empty_vals and all(is_footer_label(v) for v in non_empty_vals):
            for row in data_rows:
                if j < len(row) and j - 1 < len(row):
                    v = clean_cell(row[j])
                    if is_footer_label(v) and clean_cell(row[j - 1]) == "":
                        row[j - 1] = row[j]
                        row[j] = ""
            after_vals = [clean_cell(row[j]) if j < len(row) else "" for row in data_rows]
            if all(v == "" for v in after_vals):
                to_delete.add(j)

    for j in range(n - 1):
        cur_h = header[j]
        next_h = header[j + 1]
        looks_like_pseudo = (cur_h == "" or (j > 0 and cur_h != "" and cur_h == header[j - 1]))
        next_is_real = (next_h != "")

        if looks_like_pseudo and next_is_real:
            moved = False
            for row in data_rows:
                cur_v = clean_cell(row[j]) if j < len(row) else ""
                next_v = clean_cell(row[j + 1]) if j + 1 < len(row) else ""
                if cur_v != "" and next_v == "" and is_numeric_like(cur_v):
                    row[j + 1] = row[j]
                    row[j] = ""
                    moved = True
            after_vals = [clean_cell(row[j]) if j < len(row) else "" for row in data_rows]
            if moved and all(v == "" or is_footer_label(v) for v in after_vals):
                to_delete.add(j)

    keep_idx = [i for i in range(n) if i not in to_delete]
    new_header = [header[i] for i in keep_idx]
    new_rows = [[row[i] if i < len(row) else "" for i in keep_idx] for row in data_rows]
    return new_header, new_rows


def is_balance_like_table(table_name: str) -> bool:
    s = normalize(table_name)
    balance_aliases = {
        "balance_sheet",
        "合并资产负债表",
        "合并年初到报告期末资产负债表",
    }
    return s in balance_aliases or "资产负债表" in s


def classify_period_label(col_name: str, table_name: str = "") -> tuple[int, str]:
    s = normalize(col_name)
    table_name = normalize(table_name)

    if is_balance_like_table(table_name):
        balance_rules = [
            (220, ["本报告期末", "报告期末", "期末余额", "期末", "调整后"]),
            (210, ["本期末", "调整前"]),
            (200, ["上年度末", "上年末", "年初余额", "期初余额", "期初", "上期期末"]),
            (190, ["年末余额", "年末"]),
        ]
        for score, patterns in balance_rules:
            if any(p in s for p in patterns):
                return score, s
    else:
        flow_rules = [
            (220, ["年初至报告期末", "本报告期累计", "本年累计", "本期累计"]),
            (210, ["上年初至上年报告期末", "上年同期累计", "上年累计", "同期累计"]),
            (205, ["本报告期", "本期发生额", "本期"]),
            (195, ["上年同期", "上期发生额", "上期"]),
            (185, ["本报告期末"]),
            (175, ["上年度末", "上年末"]),
        ]
        for score, patterns in flow_rules:
            if any(p in s for p in patterns):
                return score, s

    year_match = re.search(r"(20\d{2})", s)
    if year_match:
        year = int(year_match.group(1))
        if is_balance_like_table(table_name):
            if any(k in s for k in ["12月31日", "年末", "期末", "期末余额", "本报告期末"]):
                return 100 + year, s
            if any(k in s for k in ["年初", "期初", "上年度末"]):
                return 80 + year, s
            return 60 + year, s

        if any(k in s for k in ["年度", "1-12月", "年初至", "累计"]):
            return 100 + year, s
        if any(k in s for k in ["本报告期", "上年同期", "1-3月", "1-6月", "1-9月"]):
            return 90 + year, s
        return 70 + year, s

    return -1, s

def find_period_columns(df, table_name: str = "", top_n=2, annual_mode=False):
    candidates = []

    for idx, col in enumerate(df.columns):
        col_str = str(col)
        s = normalize(col_str)

        # 跳过备注列、附注列、同比增减列
        if "附注" in s or "备注" in s:
            continue
        if is_change_column(col_str):
            continue

        # 年报模式下跳过季度列
        if annual_mode and is_quarter_column(col_str):
            continue

        score, label = classify_period_label(col_str, table_name)
        if score < 0:
            continue

        if annual_mode and not is_annual_column(col_str, table_name):
            continue

        m = re.search(r"(20\d{2})", s)
        if not m:
            continue

        year = int(m.group(1))

        candidates.append({
            "col": col_str,
            "idx": idx,
            "year": year,
            "score": score,
            "adj_priority": get_adjust_priority(col_str),
        })

    if not candidates:
        return []

    # 先按 年份倒序，再按 调整优先级，再按匹配分数
    candidates = sorted(
        candidates,
        key=lambda x: (x["year"], x["adj_priority"], x["score"]),
        reverse=True
    )

    # 每个年份只取一个，确保拿到 本年 + 上年
    picked = []
    used_years = set()

    for item in candidates:
        if item["year"] in used_years:
            continue
        picked.append(item["col"])
        used_years.add(item["year"])
        if len(picked) >= top_n:
            break

    return reorder_period_columns(picked, df.columns)
# def find_period_columns(df, table_name: str = "", top_n=2, annual_mode=False):
#     candidates = []

#     for col in df.columns:
#         col_str = str(col)

#         if annual_mode and is_quarter_column(col_str):
#             continue

#         score, label = classify_period_label(col_str, table_name)
#         if score < 0:
#             continue

#         if annual_mode and not is_annual_column(col_str, table_name):
#             continue

#         candidates.append({
#             "col": col_str,
#             "score": score,
#             "year_key": get_year_key(col_str),
#             "adj_priority": get_adjust_priority(col_str),
#         })

#     selected = sorted(
#         candidates,
#         key=lambda x: (
#             x["year_key"],        # 年份优先
#             x["adj_priority"],    # 普通 > 调整后 > 调整前
#             x["score"]            # 期别匹配度
#         ),
#         reverse=True
#     )

#     return [x["col"] for x in selected[:top_n]]

In [9]:
# def merge_same_value_rows(df):
#     if df.empty:
#         return df

#     rows = df.copy().reset_index(drop=True)
#     result = []
#     i = 0

#     while i < len(rows):
#         base = rows.iloc[i].copy()
#         j = i + 1

#         while j < len(rows):
#             curr = rows.iloc[j]

#             same_except_col0 = True
#             for c in range(1, rows.shape[1]):
#                 if clean_cell(base.iloc[c]) != clean_cell(curr.iloc[c]):
#                     same_except_col0 = False
#                     break

#             if same_except_col0:
#                 base.iloc[0] = clean_cell(base.iloc[0]) + clean_cell(curr.iloc[0])
#                 j += 1
#                 continue

#             if rows.shape[1] > 1:
#                 same_except_col1 = True
#                 for c in range(rows.shape[1]):
#                     if c == 1:
#                         continue
#                     if clean_cell(base.iloc[c]) != clean_cell(curr.iloc[c]):
#                         same_except_col1 = False
#                         break

#                 if same_except_col1:
#                     base.iloc[1] = clean_cell(base.iloc[1]) + clean_cell(curr.iloc[1])
#                     j += 1
#                     continue

#             break

#         result.append(base.tolist()) 
#         i = j

#     return pd.DataFrame(result, columns=df.columns)
def merge_same_value_rows(df):
    if df.empty:
        return df

    # 先把所有空白统一成 ''
    rows = df.copy().reset_index(drop=True)

    def norm(x):
        if pd.isna(x):
            return ''
        s = str(x).replace('\xa0', ' ').strip()
        return '' if s.lower() == 'nan' else s

    rows = rows.map(norm)

    result = []
    i = 0

    while i < len(rows):
        base = rows.iloc[i].copy()
        j = i + 1

        while j < len(rows):
            curr = rows.iloc[j]

            # 1) 除第0列外都相同 -> 合并第0列
            same_except_col0 = all(
                base.iloc[c] == curr.iloc[c]
                for c in range(1, rows.shape[1])
            )
            if same_except_col0:
                base.iloc[0] = base.iloc[0] + curr.iloc[0]
                j += 1
                continue

#             2) 下一行只有某一列非空，其余全空 -> 视为上一行该列的续行
            merged = False
            for name_col in range(rows.shape[1]):
                curr_only_name = (
                    curr.iloc[name_col] != '' and
                    all(curr.iloc[c] == '' for c in range(rows.shape[1]) if c != name_col)
                )
                base_has_other_data = any(
                    base.iloc[c] != '' for c in range(rows.shape[1]) if c != name_col
                )

                if curr_only_name and base_has_other_data:
                    base.iloc[name_col] = base.iloc[name_col] + curr.iloc[name_col]
                    j += 1
                    merged = True
                    break

            if merged:
                continue

#             # 2) 下一行只有前两列中的某一列非空，其余全空 -> 视为上一行该列的续行
#             merged = False
#             for name_col in [0, 2]:
#                 curr_only_name = (
#                     curr.iloc[name_col] != '' and
#                     all(curr.iloc[c] == '' for c in range(rows.shape[1]) if c != name_col)
#                 )
#                 base_has_other_data = any(
#                     base.iloc[c] != '' for c in range(rows.shape[1]) if c != name_col
#                 )

#                 if curr_only_name and base_has_other_data:
#                     base.iloc[name_col] = base.iloc[name_col] + curr.iloc[name_col]
#                     j += 1
#                     merged = True
#                     break

#             if merged:
#                 continue

            # 3) 除第1列外都相同 -> 合并第1列
            if rows.shape[1] > 1:
                same_except_col1 = all(
                    base.iloc[c] == curr.iloc[c]
                    for c in range(rows.shape[1]) if c != 1
                )
                if same_except_col1:
                    base.iloc[1] = base.iloc[1] + curr.iloc[1]
                    j += 1
                    continue

            break

        result.append(base.tolist())
        i = j

    return pd.DataFrame(result, columns=df.columns)

In [10]:
def normalize_header(row):
    if row is None:
        return []
    return [normalize(x) for x in row]

def table_near_page_bottom(page, table_obj, margin=100):
    """
    判断表格是否靠近页底
    bbox = (x0, top, x1, bottom)
    """
    _, _, _, bottom = table_obj.bbox
    return (page.height - bottom) <= margin

def table_near_page_top(page, table_obj, margin=100):
    """
    判断表格是否靠近页顶
    """
    _, top, _, _ = table_obj.bbox
    return top <= margin

def similar_horizontal_span(table0_obj, table1_obj, tol=20):
    """
    判断两张表左右边界是否接近
    同一张续表通常 x0/x1 很接近
    """
    x0_0, _, x1_0, _ = table0_obj.bbox
    x0_1, _, x1_1, _ = table1_obj.bbox

    return abs(x0_0 - x0_1) <= tol and abs(x1_0 - x1_1) <= tol

def can_merge_tables_by_bbox(df1: pd.DataFrame, df2: pd.DataFrame,
                             page0, table0_obj,
                             page1, table1_obj) -> bool:
    """
    跨页合并判断：
    1. 列数一致
    2. page0 最后一张表靠近页底
    3. page1 第一张表靠近页顶
    4. 两张表左右边界接近
    """
    if df1.empty or df2.empty:
        return False

    # 1. 列数必须一致
#     if df1.shape[1] != df2.shape[1]:
#         return False
    if abs(df1.shape[1] - df2.shape[1]) > 2:
        return False

    # 2. page0 尾表必须靠近页底
    if not table_near_page_bottom(page0, table0_obj, margin=100):
        return False

    # 3. page1 首表必须靠近页顶
    if not table_near_page_top(page1, table1_obj, margin=100):
        return False

    # 4. 左右边界要接近
    if not similar_horizontal_span(table0_obj, table1_obj, tol=20):
        return False

    return True

In [11]:
def merge_from_tail_table_bbox(pdf, start_page_idx, base_df, base_table_obj):
    """
    从当前页最后一张表开始，尝试与后续页第一张表跨页合并
    依赖 bbox 坐标判断
    """
    current_df = base_df.copy()
    current_table_obj = base_table_obj
    current_page = pdf.pages[start_page_idx]

    consumed_tables = []
    j = start_page_idx + 1

    while j < len(pdf.pages):
        next_page = pdf.pages[j]
        next_table_objs = next_page.find_tables()

        if not next_table_objs:
            break

        # 只看下一页第一张表
        next_table_obj = next_table_objs[0]
        next_table = next_table_obj.extract()

        if not next_table or len(next_table) < 2:
            break

        next_df = pd.DataFrame(next_table).map(normalize)

        # 用 bbox + 列数 判断是否应合并
        if not can_merge_tables_by_bbox(
            current_df, next_df,
            current_page, current_table_obj,
            next_page, next_table_obj
        ):
            break

        h1 = normalize_header(current_df.iloc[0].tolist())
        h2 = normalize_header(next_df.iloc[0].tolist())

        # 如果下一页重复了表头，则去掉首行
        if h1 == h2:
            next_df = next_df.iloc[1:].reset_index(drop=True)

        current_df = pd.concat([current_df, next_df], ignore_index=True)
        consumed_tables.append((j, 0))

        # 更新“当前页 / 当前表对象”，方便继续往后判断
        current_page = next_page
        current_table_obj = next_table_obj

        # 只有下一页只有这一张表，才继续往后看
        if len(next_table_objs) == 1:
            j += 1
        else:
            break

    return current_df, consumed_tables

In [12]:
def is_unit_fragment(s: str) -> bool:
    s = normalize(s)
    if re.fullmatch(r"[\(（][^()（）]{1,12}[\)）]", s):
        return True
    return s in {"元", "万元", "%", "元/股"}


def seems_complete_label(s: str) -> bool:
    s = normalize(s)
    if s.endswith(")") or s.endswith("）"):
        return True

    complete_suffixes = [
        "营业收入", "营业总收入", "净利润", "利润", "净额", "收益", "收益率", "每股收益",
        "现金流量净额", "货币资金", "应收账款", "存货", "金融资产", "在建工程",
        "资产总计", "总资产", "应付账款", "预收款项", "预收账款", "负债合计",
        "总负债", "合同负债", "短期借款", "未分配利润", "股东权益合计",
        "营业成本", "销售费用", "管理费用", "财务费用", "研发费用",
        "税金及附加", "营业利润", "利润总额", "其他收益", "资产减值损失", "信用减值损失"
    ]
    return any(s.endswith(x) for x in complete_suffixes)

def find_multiline_matched_row(
    df_idx: pd.DataFrame,
    kw_list_norm: list[str],
    period_cols: list[str],
    max_merge_rows: int = 8
):
    kw_list_norm = sorted([normalize(k) for k in kw_list_norm], key=len, reverse=True)
    row_names = [normalize(x) for x in df_idx.index.tolist()]

    def same_values(i: int, j: int) -> bool:
        return all(
            clean_cell(df_idx.iloc[i][col]) == clean_cell(df_idx.iloc[j][col])
            for col in period_cols
        )

    def get_score(candidate: str):
        candidate = normalize(candidate)

        # 完全匹配优先
        for kw in kw_list_norm:
            if candidate == kw:
                return (0, 0, -len(kw))

        # 包含匹配次之
        contain_kws = [kw for kw in kw_list_norm if kw in candidate]
        if contain_kws:
            best_kw = max(contain_kws, key=len)
            extra_len = len(candidate) - len(best_kw)
            return (1, extra_len, -len(best_kw))

        return None

    best_candidate = None

    for i, row_name in enumerate(row_names):
        if row_name == "":
            continue

        combined = row_name
        score = get_score(combined)
        if score is not None:
            candidate = (score, combined, i)
            if best_candidate is None or candidate[0] < best_candidate[0]:
                best_candidate = candidate

        for j in range(i + 1, min(len(row_names), i + max_merge_rows)):
            nxt = row_names[j]
            if nxt == "" or not same_values(i, j):
                break

            if not is_unit_fragment(nxt) and seems_complete_label(combined):
                break

            combined += nxt
            score = get_score(combined)
            if score is not None:
                candidate = (score, combined, i)
                if best_candidate is None or candidate[0] < best_candidate[0]:
                    best_candidate = candidate

    if best_candidate is None:
        return None, None

    return best_candidate[1], best_candidate[2]

In [13]:
def extract_from_df(
    df: pd.DataFrame,
    table_names: list[str],
    key_words: dict,
    annual_mode: bool = False
) -> Dict[str, Any]:
    result: Dict[str, Any] = {}

    first_col = df.columns[0]
    df_idx = df.set_index(first_col)
    df_idx.index = df_idx.index.map(normalize)

    for table_name in table_names:
        indicators = key_words.get(table_name, {})

        period_cols = find_period_columns(
            df_idx,
            table_name=table_name,
            top_n=2,
            annual_mode=annual_mode
        )

        if not period_cols:
            if annual_mode:
                continue
            fallback_cols = [
                c for c in df_idx.columns
                if "附注" not in str(c) and "备注" not in str(c)
            ]
            period_cols = fallback_cols[:2]
            
        period_cols = reorder_period_columns(period_cols, df_idx.columns)

        for indicator_name, kw_list in indicators.items():
            kw_list_norm = [normalize(x) for x in kw_list]

            matched_row, matched_pos = find_multiline_matched_row(
                df_idx=df_idx,
                kw_list_norm=kw_list_norm,
                period_cols=period_cols,
                max_merge_rows=8
            )

            if matched_pos is None:
                continue

            row_series = df_idx.iloc[matched_pos][period_cols]
            
            values = {
                str(col): parse_numeric_value(row_series[col])
                for col in period_cols
            }

            key = f"{table_name}.{indicator_name}"
            result[key] = {
                "matched_row": matched_row,
                "columns": [str(c) for c in period_cols],
                "values": values
            }

    return result

In [14]:
def find_cell_span(table_cells):
    """
    获取输入表的所有合并单元格情况，
    返回值形式：
    {
      "row": [[row_no, col_start, col_end], ...],   # 横向合并
      "col": [[col_no, row_start, row_end], ...]    # 纵向合并
    }
    """
    result = {"row": [], "col": []}
    x = sorted(set(i[0] for i in table_cells).union(set(i[2] for i in table_cells)))
    y = sorted(set(i[1] for i in table_cells).union(set(i[3] for i in table_cells)))

    for item in table_cells:
        x0, y0, x1, y1 = item[0], item[1], item[2], item[3]
        index_x0 = x.index(x0)
        index_x1 = x.index(x1)
        index_y0 = y.index(y0)
        index_y1 = y.index(y1)

        if index_x1 - index_x0 > 1:
            for i in range(index_y0, index_y1):
                result["row"].append([i, index_x0, index_x1 - 1])
        if index_y1 - index_y0 > 1:
            for i in range(index_x0, index_x1):
                result["col"].append([i, index_y0, index_y1 - 1])

    return result



In [15]:
def get_header_index(table_text, cell_span_result):
    """
    逻辑：默认表头至少2列值不为空，如果不符合条件就往下一行找。
    找到表头第一行后，若存在从该行开始的纵向合并，则其最大 row_end 作为表头结束行。
    返回 [index_start, index_end]
    """
    index_start = 0

    for item in table_text:
        if len(item) - item.count(None) < 2:
            index_start += 1
        else:
            break

    index_end = index_start
    if cell_span_result['col']:
        for col_span in cell_span_result['col']:
            row_start = col_span[1]
            row_end = col_span[2]
            if row_start == index_start and row_end >= index_end:
                index_end = row_end

#     print('智能识别的表头开始结束下标为：', index_start, index_end, '其他页内容开始下标为：', index_end - index_start + 1)
    return [index_start, index_end]



In [16]:
def set_span_cell_all(table_text, cell_span_result, content_index):
    """
    将表头中的合并单元格拆开，并将值填充到每个单元格。
    """
    len_table = len(table_text)
    if cell_span_result['col']:
        for col_span in cell_span_result['col']:
            col, row_start, row_end = col_span
            if 0 <= row_start - content_index < len(table_text):
                if col < len(table_text[row_start - content_index]):
                    col_value = table_text[row_start - content_index][col]
                    for j in range(row_start, row_end + 1):
                        row_idx = j - content_index
                        if 0 <= row_idx < len(table_text) and col < len(table_text[row_idx]):
                            table_text[row_idx][col] = col_value

    if cell_span_result['row']:
        for row_span in cell_span_result['row']:
            row, col_start, col_end = row_span
            if 0 <= row - content_index < len(table_text):
                row_data = table_text[row - content_index]
                if col_start < len(row_data):
                    row_value = row_data[col_start]
                    for j in range(col_start, col_end + 1):
                        if j < len(row_data):
                            row_data[j] = row_value
    return table_text



In [17]:
def get_header(table_text):
    """
    根据提取的表头数据生成最终表头：
    - 先把每一行补齐到相同长度
    - 同一列从上到下收集非空值
    - 非空值之间用 _ 拼接
    - 忽略空值，避免出现尾部多余下划线
    """
    table_text = normalize_rows(table_text)
    if not table_text:
        return []

    col_num = max(len(r) for r in table_text)
    header = []

    for j in range(col_num):
        parts = []
        for i in range(len(table_text)):
            val = clean_cell(table_text[i][j])
            if val and val not in parts:
                parts.append(val)
        header.append("_".join(parts))

    return header



In [18]:
def set_span_cell_special(table_text, cell_span_result, index_col_name, content_index):
    """
    正文处理：
    1. 处理纵向合并（行合并）
    2. 处理横向合并（列合并）：若目标列在合并区间内，则将值写入目标列，否则保留起始列
    """
    table_text = normalize_rows(table_text)
    len_table = len(table_text)
    result = []

    if cell_span_result['col']:
        for col_span in cell_span_result['col']:
            col, row_start, row_end = col_span
            if 0 <= row_start - content_index < len_table:
                if col < len(table_text[row_start - content_index]):
                    col_value = table_text[row_start - content_index][col]
                    for j in range(row_start, row_end + 1):
                        row_idx = j - content_index
                        if 0 <= row_idx < len_table and col < len(table_text[row_idx]):
                            table_text[row_idx][col] = col_value

    row_span_map = {}
    if cell_span_result['row']:
        for item in cell_span_result['row']:
            row_span_map[item[0]] = item

    for row_idx, row in enumerate(table_text, 1):
        current_row_no = row_idx + content_index - 1
        cleaned_row = [clean_cell(cell) for cell in row]

        if current_row_no in row_span_map:
            _, col_start, col_end = row_span_map[current_row_no]
            if col_start < len(cleaned_row):
                merged_value = cleaned_row[col_start]
                for j in range(col_start, min(col_end + 1, len(cleaned_row))):
                    cleaned_row[j] = ""

                if index_col_name != -1 and col_start <= index_col_name <= col_end and index_col_name < len(cleaned_row):
                    cleaned_row[index_col_name] = merged_value
                else:
                    cleaned_row[col_start] = merged_value

        result.append(cleaned_row)

    return result

In [19]:
def pretty_print_result(one_table_result):
    for k, v in one_table_result.items():
        table_name, indicator = k.split(".", 1)
        matched_row = v.get("matched_row", "")
        values = v.get("values", {})
        cols = v.get("columns", list(values.keys()))
        col_str = " | ".join([f"{c}={values.get(c, '')}" for c in cols])
#         print(f"[{table_name}] {indicator}  <--  行名: {matched_row}  |  {col_str}")

In [20]:
def build_dataframe(column, name):
#     return pd.DataFrame(index=column[name])
    return pd.DataFrame(index=column[name], columns=["now", "last_year"], dtype=object)

In [21]:
period_map = {
    "半年度": "HY",
    "年度": "FY",
    "一季度": "Q1",
    "三季度": "Q3",
}

def extract_shenjiao_filename(file_path, folder_name):
    file_name = os.path.basename(file_path)
    name = os.path.splitext(file_name)[0]
    name, time = name.split("：")
    year = time[:4]
    
    with pdfplumber.open(file_path) as pdf:
        first_page = pdf.pages[0]
        text = first_page.extract_text()
        
        code_match = re.search(r"(?:证券代码|公司代码)[:：]\s*(\d+)", text)
        name_match = re.search(r"(?:证券简称|公司简称)[:：]\s*([^\s]+)", text)

        stock_code = code_match.group(1) if code_match else None
        stock_name = name_match.group(1) if name_match else None
    
    for zn_period, en_period in period_map.items():
        if zn_period in time:
            return{
                "stock_abbr": stock_name,
                "stock_code": stock_code,
                "report_year": year,
                "report_period": en_period,
            }

def extract_shangjiao_filename(file_path, folder_name):
    file_name = os.path.basename(file_path)
    name = os.path.splitext(file_name)[0]
    num, time, _ = name.split("_")
    year = time[:4]
    month = time[4:6]
        
    with pdfplumber.open(file_path) as pdf:
        first_page = pdf.pages[0]
        text = first_page.extract_text()
        
        name_match = re.search(r"(?:证券简称|公司简称)[:：]\s*([^\s]+)", text)

        stock_code = code_match.group(1) if code_match else None
        stock_name = name_match.group(1) if name_match else None
        
        for zn_period, en_period in period_map.items():
            if zn_period in text:
                return {
                    "stock_abbr": name_match,
                    "stock_code": num,
                    "report_year": year,
                    "report_period": en_period,
                }

In [22]:
def extract_first_number(row):
    """提取该行第一个数字"""
    for cell in row:
        if not cell:
            continue

        cell = cell.replace(",", "")

        m = re.search(r"\d+(\.\d+)?", cell)
        if m:
            return m.group()

    return None

def clean_cell_1(text):
    """清洗单元格"""
    if text is None:
        return ""

    text = text.strip()

    # 合并数字换行
    text = re.sub(r'(?<=\d)\s*\n\s*(?=\d)', '', text)

    # 其他换行转空格
    text = re.sub(r'\n+', ' ', text)

    # 去多余空格
    text = re.sub(r'\s+', ' ', text)

    return text


def normalize_1(text):
    """用于匹配标题"""
    return re.sub(r'\s+', '', text)

In [23]:
def process_df_without_Q3(df_core, df_balance, df_cash, df_income, df_other):
    # df_core
    # operating_revenue_yoy_growth
    total_operating_revenue_last = df_core.loc["total_operating_revenue", "last_year"]
    total_operating_revenue_now = df_core.loc["total_operating_revenue", "now"]
    if isinstance(total_operating_revenue_last, str) and "%" in total_operating_revenue_last:
        df_core.loc["operating_revenue_yoy_growth", "now"] = total_operating_revenue_last
    else:
        res = (total_operating_revenue_now-total_operating_revenue_last)/total_operating_revenue_last*100
        df_core.loc["operating_revenue_yoy_growth", "now"] = f"{res:.2f}%"
        
    # net_profit_yoy_growth
    net_profit_10k_yuan_last = df_core.loc["net_profit_10k_yuan", "last_year"]
    net_profit_10k_yuan_now = df_core.loc["net_profit_10k_yuan", "now"]
    if isinstance(net_profit_10k_yuan_last, str) and "%" in net_profit_10k_yuan_last:
        df_core.loc["net_profit_yoy_growth", "now"] = net_profit_10k_yuan_last
    else:
        res = (net_profit_10k_yuan_now-net_profit_10k_yuan_last)/net_profit_10k_yuan_last*100
        df_core.loc["net_profit_yoy_growth", "now"] = f"{res:.2f}%"
    
    # net_profit_excl_non_recurring_yoy
    net_profit_excl_non_recurring_last = df_core.loc["net_profit_excl_non_recurring", "last_year"]
    net_profit_excl_non_recurring_now = df_core.loc["net_profit_excl_non_recurring", "now"]
    if isinstance(net_profit_excl_non_recurring_last, str) and "%" in net_profit_excl_non_recurring_last:
        df_core.loc["net_profit_excl_non_recurring_yoy", "now"] = net_profit_excl_non_recurring_last
    else:
        res = (net_profit_excl_non_recurring_now-net_profit_excl_non_recurring_last)/net_profit_excl_non_recurring_last*100
        df_core.loc["net_profit_excl_non_recurring_yoy", "now"] = f"{res:.2f}%"
        
    # net_asset_per_share
    guben_value = df_other.loc["total_guben_value", "now"]
    jingzichan = float(df_other.loc["total_jingzichan", "now"])
    net_asset_per_share = jingzichan/guben_value
    df_core.loc["net_asset_per_share", "now"] = round(net_asset_per_share, 2)
    
    # operating_cf_per_share
    operating_cf_net_amount = df_cash.loc["operating_cf_net_amount", "now"]
    df_core.loc["operating_cf_per_share", "now"] = round(operating_cf_net_amount/guben_value, 2)
    
    # gross_profit_margin
    val_gain = df_core.loc["total_operating_revenue", "now"]
    val_cost = df_income.loc["operating_expense_cost_of_sales", "now"]
    val_res = (val_gain-val_cost)/val_gain*100
    df_core.loc["gross_profit_margin", "now"] = f"{val_res:.2f}%"
    
    # net_profit_margin
    val_gain = df_core.loc["total_operating_revenue", "now"]
    val_gain1 = df_core.loc["net_profit_10k_yuan", "now"]
    val_res = val_gain1/val_gain*100
    df_core.loc["net_profit_margin", "now"] = f"{val_res:.2f}%"
    
    # roe_weighted_excl_non_recurring
    ###
    
    # df_balance
    # asset_total_assets_yoy_growth
    asset_total_assets_now = df_balance.loc["asset_total_assets", "now"]
    asset_total_assets_last = df_balance.loc["asset_total_assets", "last_year"]
    if isinstance(asset_total_assets_last, str) and "%" in asset_total_assets_last:
        df_balance.loc["asset_total_assets_yoy_growth", "now"] = asset_total_assets_last
    else:
        asset_total_assets_yoy_growth = (asset_total_assets_now - asset_total_assets_last)/asset_total_assets_last*100
        df_balance.loc["asset_total_assets_yoy_growth", "now"] = f"{asset_total_assets_yoy_growth:.2f}%"
        
    # liability_advance_from_customers
    if df_balance.loc["liability_advance_from_customers", "now"] == None:
        df_balance.loc["liability_advance_from_customers", "now"] = 0.0
        
    # liability_total_liabilities_yoy_growth
    liability_total_liabilities_now = df_balance.loc["liability_total_liabilities", "now"]
    liability_total_liabilities_last = df_balance.loc["liability_total_liabilities", "last_year"]
    if isinstance(liability_total_liabilities_last, str) and "%" in liability_total_liabilities_last:
        df_balanca.loc["liability_total_liabilities_yoy_growth", "now"] = liability_total_liabilities_yoy_growth_last
    else:
        liability_total_liabilities_yoy_growth = (liability_total_liabilities_now - liability_total_liabilities_last)/liability_total_liabilities_last*100
        df_balance.loc["liability_total_liabilities_yoy_growth", "now"] = f"{liability_total_liabilities_yoy_growth:.2f}%"
        
    # asset_liability_ratio
    liability_total_liabilities = df_balance.loc["liability_total_liabilities", "now"]
    asset_total_assets = df_balance.loc["asset_total_assets", "now"]
    asset_liability_ratio = liability_total_liabilities/asset_total_assets*100
    df_balance.loc["asset_liability_ratio", "now"] = f"{asset_liability_ratio:.2f}%"
    
    # df_cash
    # net_cash_flow_yoy_growth
    net_cash_flow_now = df_cash.loc["net_cash_flow", "now"]
    net_cash_flow_last = df_cash.loc["net_cash_flow", "last_year"]
    if isinstance(net_cash_flow_last, str) and "%" in net_cash_flow_last:
        df_cash.loc["net_cash_flow_yoy_growth", "now"] = net_cash_flow_last
    else:
        net_cash_flow_yoy_growth = (net_cash_flow_now - net_cash_flow_last)/net_cash_flow_last
        df_cash.loc["net_cash_flow_yoy_growth", "now"] = f"{net_cash_flow_yoy_growth:.2f}%"
        
    # operating_cf_ratio_of_net_cf
    operating_cf_net_amount = df_cash.loc["operating_cf_net_amount", "now"]
    net_cash_flow = df_cash.loc["net_cash_flow", "now"]
    operating_cf_ratio_of_net_cf = operating_cf_net_amount/net_cash_flow*100
    df_cash.loc["operating_cf_ratio_of_net_cf", "now"] = f"{operating_cf_ratio_of_net_cf:.2f}%"
    
    # investing_cf_ratio_of_net_cf
    investing_cf_net_amount = df_cash.loc["investing_cf_net_amount", "now"]
    investing_cf_ratio_of_net_cf = investing_cf_net_amount/net_cash_flow*100
    df_cash.loc["investing_cf_ratio_of_net_cf", "now"] = f"{investing_cf_ratio_of_net_cf:.2f}%"
    
    # financing_cf_ratio_of_net_cf
    financing_cf_net_amount = df_cash.loc["financing_cf_net_amount", "now"]
    financing_cf_ratio_of_net_cf = financing_cf_net_amount/net_cash_flow*100
    df_cash.loc["financing_cf_ratio_of_net_cf", "now"] = f"{financing_cf_ratio_of_net_cf:.2f}%"
    
    # df_income
    # net_profit_yoy_growth
    net_profit_now = df_income.loc["net_profit", "now"]
    net_profit_last = df_income.loc["net_profit", "last_year"]
    if isinstance(net_profit_last, str) and "%" in net_profit_last:
        df_income.loc["net_profit_yoy_growth", "now"] = net_profit_last
    else:
        net_profit_yoy_growth = (net_profit_now - net_profit_last)/net_profit_last*100
        df_income.loc["net_profit_yoy_growth", "now"] = f"{net_profit_yoy_growth:.2f}%"
        
    # operating_revenue_yoy_growth
    total_operating_revenue_now = df_income.loc["total_operating_revenue", "now"]
    total_operating_revenue_last = df_income.loc["total_operating_revenue", "last_year"]
    if isinstance(total_operating_revenue_last, str) and "%" in total_operating_revenue_last:
        df_income.loc["operating_revenue_yoy_growth", "now"] = total_operating_revenue_last
    else:
        operating_revenue_yoy_growth = (total_operating_revenue_now - total_operating_revenue_last)/total_operating_revenue_last*100
        df_income.loc["operating_revenue_yoy_growth", "now"] = f"{operating_revenue_yoy_growth:.2f}%"

In [24]:
def process_one_shenjiao_pdf_without_Q3(pdf_path):
    consumed = set()
    
    df_core = build_dataframe(table_columns, "core_performance_indicators_sheet")
    df_balance = build_dataframe(table_columns, "balance_sheet")
    df_cash = build_dataframe(table_columns, "cash_flow_sheet")
    df_income = build_dataframe(table_columns, "income_sheet")
    df_other = build_dataframe(table_columns, "other")
    
    df_map = {
        "core_performance_indicators_sheet": df_core,
        "balance_sheet": df_balance,
        "cash_flow_sheet": df_cash,
        "income_sheet": df_income,
        "other": df_other,
    }
    
    file_name = os.path.basename(pdf_path)
    if "摘要" in file_name:
        return None
    
    folder_name = os.path.basename(os.path.dirname(pdf_path))
    dic = extract_shenjiao_filename(pdf_path, folder_name)
    report_period = dic["report_period"]
    if report_period == "Q3":
        return None
    
    with pdfplumber.open(pdf_path) as pdf:
        
        for tot_name, table in df_map.items():
            cur_table = df_map[tot_name]
            for key, val in dic.items():
                cur_table.loc[key, "now"] = val
                
        for page_idx, page in enumerate(pdf.pages):
            text = page.extract_text()
            tables = page.find_tables()
            
            page_text = normalize(text or "")
            
            for table_idx, table_obj in enumerate(tables):
                if (page_idx, table_idx) in consumed:
                    continue
                    
                table = table_obj.extract()
                if not table or len(table) < 2:
                    continue
                    
                table_text_for_match = normalize(str(table))
                
                matches = [k for k, aliases in table_name.items() if any(normalize(alias) in page_text or normalize(alias) in table_text_for_match for alias in aliases)]
                if not matches:
                    continue
                    
                skip_matches = [k for k in skip_words if normalize(k) in table_text_for_match]
                if skip_matches:
                    continue
                    
                df = pd.DataFrame(table).map(normalize)
                
                if table_idx == len(tables) - 1:
                    merged_df, consumed_tables = merge_from_tail_table_bbox(pdf, page_idx, df, table_obj)
                    
#                     if not merged_df.equals(df):
#                         print("="*100)
#                         print("Page: ", page_idx+1)
#                         print("from")
#                         print(merged_df)
#                         print("end")
#                         print("="*100)
                    
                    df = merged_df
                    consumed.update(consumed_tables)
                    
                table_text = normalize_rows(df.values.tolist())
                if len(table_text) < 2:
                    continue
                    
                cell_span_result =find_cell_span(table_obj.cells)
                header_index_array = get_header_index(table_text, cell_span_result)
                header_start, header_end = header_index_array
                content_index = header_end + 1
                
                if content_index >= len(table_text):
                    continue
                    
                header_part = normalize_rows(table_text[header_start:content_index])
                header_result = set_span_cell_all(header_part, cell_span_result, header_start)
                header = get_header(header_result)
                index_col_name = find_special_column_index(header)

                data_rows = normalize_rows(table_text[content_index:])
                result_rows = set_span_cell_special(data_rows, cell_span_result, index_col_name, content_index)
                header, result_rows = fix_misaligned_year_columns(header, result_rows)
                header, result_rows = fix_shifted_columns(header, result_rows)
                header, result_rows = drop_empty_columns(header, result_rows)

                if not header or not result_rows:
                    continue

                table_df = pd.DataFrame(result_rows, columns=header)
                
                table_df = table_df.fillna('').astype(str) ###
                table_df = table_df.map(lambda x: x.replace('\xa0', ' ').strip()) ###
                
                table_df = merge_same_value_rows(table_df)
            
                if table_df.empty:
                    continue

                annual_mode = is_annual_report(pdf_path)
                
                for _, row in table_df.iterrows():
                    row = row.tolist()
                    row_clean = [clean_cell_1(c) for c in row]
                    row_text = normalize_1("".join(row_clean))

                    if "归属于上市公司股东的净资产" in row_text or "归属于上市公司股东的所有者权益" in row_text or "归属于上市公司股东的所" in row_text or "归属于母公司所有者权益合计" in row_text:
                        first_number = extract_first_number(row_clean)
                        df_other.loc["total_jingzichan", "now"] = first_number
                        
                    if "实收资本(或股本)" in row_text or "股本"in row_text or "实收资本" in row_text:
                        first_number = extract_first_number(row_clean)
                        if first_number is not None:
                            df_other.loc["total_guben_value", "now"] = float(first_number.replace(",", ""))

                one_table_result = extract_from_df(
                    table_df,
                    matches,
                    key_words,
                    annual_mode=annual_mode
                )

                print("Page: ", page_idx+1)
                print(table_df)
            
                for k, v in one_table_result.items():
                    tablename, indicator = k.split(".", 1)
                    values = v.get("values", {})
                    vals = list(values.values())

                    current_df = df_map[tablename]
                    
#                     current_df.loc[indicator, "now"] = vals[0]
#                     current_df.loc[indicator, "last_year"] = vals[1] if len(vals) > 1 else None
                    
                    new_now = vals[0] if len(vals) > 0 else None
                    new_last = vals[1] if len(vals) > 1 else None
                    old_now = current_df.loc[indicator, "now"]
                    old_last = current_df.loc[indicator, "last_year"]
                    if old_now is None and new_now is not None:
                        current_df.loc[indicator, "now"] = new_now
                    elif old_now is None:
                        pass
                    elif new_now is not None:
                        current_df.loc[indicator, "now"] = new_now
                    if old_last is None and new_last is not None:
                        current_df.loc[indicator, "last_year"] = new_last
                    elif old_last is None:
                        pass
                    elif new_last is not None:
                        current_df.loc[indicator, "last_year"] = new_last
                    
    process_df_without_Q3(df_core, df_balance, df_cash, df_income, df_other)
    
    return {
        "df_core": df_core,
        "df_balance": df_balance,
        "df_cash": df_cash,
        "df_income": df_income,
        "df_other": df_other,
    }

In [25]:
process_one_shenjiao_pdf_without_Q3(r"B题-示例数据\示例数据\附件2：财务报告\reports-深交所\华润三九：2022年年度报告.pdf")

Page:  6
                           股票简称  \
0                     股票上市证券交易所   
1                       公司的中文名称   
2                       公司的中文简称   
3                   公司的外文名称(如有)   
4                 公司的外文名称缩写(如有)   
5                      公司的法定代表人   
6                          注册地址   
7                     注册地址的邮政编码   
8   公司注册地址历史变更情况                  
9                          办公地址   
10                    办公地址的邮政编码   
11                         公司网址   
12                         电子信箱   

                                                 华润三九 股票代码 000999  
0                                             深圳证券交易所              
1                                        华润三九医药股份有限公司              
2                                                华润三九              
3   ChinaResourcesSanjiuMedical&PharmaceuticalCo.,...              
4                                            CRSanjiu              
5                                                 赵炳祥              
6                       

Page:  101
                             预收款项                                      
0                            合同负债       1,861,577.68         279,873.41
1                          应付职工薪酬     248,379,323.78     222,257,590.92
2                            应交税费     111,135,694.24      36,332,740.57
3                     其他应付款持有待售负债   7,307,233,599.16   5,800,168,002.33
4                     一年内到期的非流动负债      10,442,155.68      17,122,103.59
5                          其他流动负债         242,005.05          36,383.55
6   流动负债合计非流动负债：长期借款应付债券其中：优先股永续债   7,973,480,558.73   6,471,542,876.96
7                       租赁负债长期应付款      24,716,502.63      29,762,762.06
8                    长期应付职工薪酬预计负债       4,680,000.00       4,530,000.00
9                            递延收益      32,693,039.18      41,903,800.59
10                 递延所得税负债其他非流动负债      77,960,265.87      68,654,863.04
11                        非流动负债合计     140,049,807.68     144,851,425.69
12                     负债合计所有者权益：   8,113,530,366.41 

Page:  164
                                   
0        股本溢价(注1)  1,167,854,337.12
1   同一控制下企业合并(注2)    462,661,362.05
2    收购少数股东权益(注3)         63,223.21
3     不丧失控制权的股权处置         84,874.35
4  股份支付计入所有者权益的金额                 -
5              其他     13,013,183.37
6                                  
Page:  164
                                        
0                 股本溢价  1,167,854,337.12
1            同一控制下企业合并    462,661,362.05
2  收购少数股东权益不丧失控制权的股权处置                 -
3                   其他     12,904,080.21
4                                       
Page:  164
                         
0  库存股  -  143,571,233.31


{'df_core':                                               now       last_year
 serial_number                                 NaN             NaN
 stock_code                                   None             NaN
 stock_abbr                                   None             NaN
 report_period                                  FY             NaN
 report_year                                  2022             NaN
 eps                                          2.48             2.1
 total_operating_revenue            18079461482.75  15544401735.35
 operating_revenue_yoy_growth               16.31%             NaN
 operating_revenue_qoq_growth                  NaN             NaN
 net_profit_10k_yuan                 2448802103.34   2055104407.14
 net_profit_yoy_growth                      19.16%             NaN
 net_profit_qoq_growth                         NaN             NaN
 net_asset_per_share                         14.56             NaN
 roe                                        15.15% 

In [26]:
def process_one_shangjiao_pdf_without_Q3(pdf_path):
    consumed = set()
    
    df_core = build_dataframe(table_columns, "core_performance_indicators_sheet")
    df_balance = build_dataframe(table_columns, "balance_sheet")
    df_cash = build_dataframe(table_columns, "cash_flow_sheet")
    df_income = build_dataframe(table_columns, "income_sheet")
    df_other = build_dataframe(table_columns, "other")
    
    df_map = {
        "core_performance_indicators_sheet": df_core,
        "balance_sheet": df_balance,
        "cash_flow_sheet": df_cash,
        "income_sheet": df_income,
        "other": df_other,
    }
    
    file_name = os.path.basename(pdf_path)
    if "摘要" in file_name:
        return None
    
    folder_name = os.path.basename(os.path.dirname(pdf_path))
    dic = extract_shangjiao_filename(pdf_path, folder_name)
    report_period = dic["report_period"]
    if report_period == "Q3":
        return None
    
    with pdfplumber.open(pdf_path) as pdf:
        
        for tot_name, table in df_map.items():
            cur_table = df_map[tot_name]
            for key, val in dic.items():
                cur_table.loc[key, "now"] = val
                
        for page_idx, page in enumerate(pdf.pages):
            text = page.extract_text()
            tables = page.find_tables()
            
            page_text = normalize(text or "")
            
            for table_idx, table_obj in enumerate(tables):
                if (page_idx, table_idx) in consumed:
                    continue
                    
                table = table_obj.extract()
                if not table or len(table) < 2:
                    continue
                    
                table_text_for_match = normalize(str(table))
                
                matches = [k for k, aliases in table_name.items() if any(normalize(alias) in page_text or normalize(alias) in table_text_for_match for alias in aliases)]
                if not matches:
                    continue
                    
                skip_matches = [k for k in skip_words if normalize(k) in table_text_for_match]
                if skip_matches:
                    continue
                    
                df = pd.DataFrame(table).map(normalize)
                
                if table_idx == len(tables) - 1:
                    merged_df, consumed_tables = merge_from_tail_table_bbox(pdf, page_idx, df, table_obj)
                    df = merged_df
                    consumed.update(consumed_tables)
                    
                table_text = normalize_rows(df.values.tolist())
                if len(table_text) < 2:
                    continue
                    
                cell_span_result =find_cell_span(table_obj.cells)
                header_index_array = get_header_index(table_text, cell_span_result)
                header_start, header_end = header_index_array
                content_index = header_end + 1
                
                if content_index >= len(table_text):
                    continue
                    
                header_part = normalize_rows(table_text[header_start:content_index])
                header_result = set_span_cell_all(header_part, cell_span_result, header_start)
                header = get_header(header_result)
                index_col_name = find_special_column_index(header)

                data_rows = normalize_rows(table_text[content_index:])
                result_rows = set_span_cell_special(data_rows, cell_span_result, index_col_name, content_index)
                header, result_rows = fix_misaligned_year_columns(header, result_rows)
                header, result_rows = fix_shifted_columns(header, result_rows)
                header, result_rows = drop_empty_columns(header, result_rows)

                if not header or not result_rows:
                    continue

                table_df = pd.DataFrame(result_rows, columns=header)
                table_df = merge_same_value_rows(table_df)
            
                if table_df.empty:
                    continue

                annual_mode = is_annual_report(pdf_path)
                
                for _, row in table_df.iterrows():
                    row = row.tolist()
                    row_clean = [clean_cell_1(c) for c in row]
                    row_text = normalize_1("".join(row_clean))

                    if "归属于上市公司股东的净资产" in row_text or "归属于上市公司股东的所有者权益" in row_text or "归属于上市公司股东的所" in row_text:
                        first_number = extract_first_number(row_clean)
                        df_other.loc["total_jingzichan", "now"] = first_number
                        
                    if "实收资本(或股本)" in row_text or "股本"in row_text or "实收资本" in row_text:
                        first_number = extract_first_number(row_clean)
                        if first_number is not None:
                            df_other.loc["total_guben_value", "now"] = float(first_number.replace(",", ""))
                            
                    

                one_table_result = extract_from_df(
                    table_df,
                    matches,
                    key_words,
                    annual_mode=annual_mode
                )

            
                for k, v in one_table_result.items():
                    tablename, indicator = k.split(".", 1)
                    values = v.get("values", {})
                    vals = list(values.values())

                    current_df = df_map[tablename]
#                     current_df.loc[indicator, "now"] = vals[0]
#                     current_df.loc[indicator, "last_year"] = vals[1] if len(vals) > 1 else None
                    
                    new_now = vals[0] if len(vals) > 0 else None
                    new_last = vals[1] if len(vals) > 1 else None
                    old_now = current_df.loc[indicator, "now"]
                    old_last = current_df.loc[indicator, "last_year"]
                    if old_now is None and new_now is not None:
                        current_df.loc[indicator, "now"] = new_now
                    elif old_now is None:
                        pass
                    elif new_now is not None:
                        current_df.loc[indicator, "now"] = new_now
                    if old_last is None and new_last is not None:
                        current_df.loc[indicator, "last_year"] = new_last
                    elif old_last is None:
                        pass
                    elif new_last is not None:
                        current_df.loc[indicator, "last_year"] = new_last
                    
    process_df_without_Q3(df_core, df_balance, df_cash, df_income, df_other)
    
    return {
        "df_core": df_core,
        "df_balance": df_balance,
        "df_cash": df_cash,
        "df_income": df_income,
        "df_other": df_other,
    }

In [27]:
shenjiao_folder = r"B题-示例数据\示例数据\附件2：财务报告\reports-深交所"

all_core = []
all_balance = []
all_cash = []
all_income = []
all_other = []

for file_name in tqdm(os.listdir(shenjiao_folder), desc="Processing"):
    if not file_name.lower().endswith(".pdf"):
        continue
        
    pdf_path = os.path.join(shenjiao_folder, file_name)
    
    res = process_one_shenjiao_pdf_without_Q3(pdf_path)
    
    if res is None:
        continue
        
    df_core_tmp = res["df_core"].copy().T
    df_core_tmp = df_core_tmp.loc[["now"]].reset_index(drop=True)
    df_core_tmp["source_file"] = file_name
    all_core.append(df_core_tmp)
    
    df_balance_tmp = res["df_balance"].copy().T
    df_balance_tmp = df_balance_tmp.loc[["now"]].reset_index(drop=True)
    df_balance_tmp["source_file"] = file_name
    all_balance.append(df_balance_tmp)
    
    df_cash_tmp = res["df_cash"].copy().T
    df_cash_tmp = df_cash_tmp.loc[["now"]].reset_index(drop=True)
    df_cash_tmp["source_file"] = file_name
    all_cash.append(df_cash_tmp)
    
    df_income_tmp = res["df_income"].copy().T
    df_income_tmp = df_income_tmp.loc[["now"]].reset_index(drop=True)
    df_income_tmp["source_file"] = file_name
    all_income.append(df_income_tmp)
    
    df_other_tmp = res["df_other"].copy().T
    df_other_tmp = df_other_tmp.loc[["now"]].reset_index(drop=True)
    df_other_tmp["source_file"] = file_name
    all_other.append(df_other_tmp)
    
core_all = pd.concat(all_core, ignore_index=True) if all_core else pd.DataFrame()
balance_all = pd.concat(all_balance, ignore_index=True) if all_balance else pd.DataFrame()
cash_all = pd.concat(all_cash, ignore_index=True) if all_cash else pd.DataFrame()
income_all = pd.concat(all_income, ignore_index=True) if all_income else pd.DataFrame()
other_all = pd.concat(all_other, ignore_index=True) if all_other else pd.DataFrame()
    
output_path = r"B题-示例数据\示例数据\附件2：财务报告\深交extract_result.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    
    core_all.to_excel(writer, sheet_name="core", index=False)
    balance_all.to_excel(writer, sheet_name="balance", index=False)
    cash_all.to_excel(writer, sheet_name="cash", index=False)
    income_all.to_excel(writer, sheet_name="income", index=False)
    other_all.to_excel(writer, sheet_name="other", index=False)

print("保存完成:", output_path)    

Processing:   0%|                                                                               | 0/18 [00:00<?, ?it/s]

Page:  6
                           股票简称  \
0                     股票上市证券交易所   
1                       公司的中文名称   
2                       公司的中文简称   
3                   公司的外文名称(如有)   
4                 公司的外文名称缩写(如有)   
5                      公司的法定代表人   
6                          注册地址   
7                     注册地址的邮政编码   
8   公司注册地址历史变更情况                  
9                          办公地址   
10                    办公地址的邮政编码   
11                         公司网址   
12                         电子信箱   

                                                 华润三九 股票代码 000999  
0                                             深圳证券交易所              
1                                        华润三九医药股份有限公司              
2                                                华润三九              
3   ChinaResourcesSanjiuMedical&PharmaceuticalCo.,...              
4                                            CRSanjiu              
5                                                 赵炳祥              
6                       

Page:  101
                             预收款项                                      
0                            合同负债       1,861,577.68         279,873.41
1                          应付职工薪酬     248,379,323.78     222,257,590.92
2                            应交税费     111,135,694.24      36,332,740.57
3                     其他应付款持有待售负债   7,307,233,599.16   5,800,168,002.33
4                     一年内到期的非流动负债      10,442,155.68      17,122,103.59
5                          其他流动负债         242,005.05          36,383.55
6   流动负债合计非流动负债：长期借款应付债券其中：优先股永续债   7,973,480,558.73   6,471,542,876.96
7                       租赁负债长期应付款      24,716,502.63      29,762,762.06
8                    长期应付职工薪酬预计负债       4,680,000.00       4,530,000.00
9                            递延收益      32,693,039.18      41,903,800.59
10                 递延所得税负债其他非流动负债      77,960,265.87      68,654,863.04
11                        非流动负债合计     140,049,807.68     144,851,425.69
12                     负债合计所有者权益：   8,113,530,366.41 

Page:  164
                                   
0        股本溢价(注1)  1,167,854,337.12
1   同一控制下企业合并(注2)    462,661,362.05
2    收购少数股东权益(注3)         63,223.21
3     不丧失控制权的股权处置         84,874.35
4  股份支付计入所有者权益的金额                 -
5              其他     13,013,183.37
6                                  
Page:  164
                                        
0                 股本溢价  1,167,854,337.12
1            同一控制下企业合并    462,661,362.05
2  收购少数股东权益不丧失控制权的股权处置                 -
3                   其他     12,904,080.21
4                                       
Page:  164
                         
0  库存股  -  143,571,233.31


Processing:   6%|███▉                                                                   | 1/18 [00:40<11:29, 40.58s/it]

...

Processing:  89%|██████████████████████████████████████████████████████████████▏       | 16/18 [03:30<00:19,  9.90s/it]

Page:  6
            股票简称                                               华润三九 股票代码  \
0      股票上市证券交易所                                            深圳证券交易所        
1        公司的中文名称                                       华润三九医药股份有限公司        
2    公司的中文简称(如有)                                               华润三九        
3    公司的外文名称(如有)  ChinaResourcesSanjiuMedical&PharmaceuticalCo.,...        
4  公司的外文名称缩写(如有)                                           CRSanjiu        
5       公司的法定代表人                                                吴文多        

  000999  
0         
1         
2         
3         
4         
5         
Page:  6
                                                               董事会秘书  \
0          姓名                                               邢健         
1  联系地址                   深圳市龙华区观湖街道观澜高新园区观清路1号华润三九医药工业园综合办公中心         
2    电话        (86)755-66853868、(86)755-83360999-392210,392209         
3          传真                          (86)755-83360999-396006         
4        电子信

Page:  77
                                   1.权益法下可转损益的其他综合收益                  \
0  2.其他债权投资公允价值变动3.金融资产重分类计入其他综合收益的金额4.其他债权投资信用减值...                   
1                  六、综合收益总额七、每股收益：(一)基本每股收益(二)稀释每股收益  787,783,851.83   

                   
0                  
1  988,957,479.93  
Page:  77
                                              项目           2025年半年度  \
0                                 一、经营活动产生的现金流量：                      
1                                 销售商品、提供劳务收到的现金  16,688,868,901.70   
2                                        收到的税费返还       9,553,835.26   
3                                 收到其他与经营活动有关的现金     680,263,640.32   
4                                     经营活动现金流入小计  17,378,686,377.28   
5                                 购买商品、接受劳务支付的现金   5,996,967,628.40   
6                                支付给职工以及为职工支付的现金   2,648,811,247.44   
7                                        支付的各项税费   1,910,734,039.66   
8                                 支付其他与经营活动有关的现金   3,961,978,520

Processing: 100%|██████████████████████████████████████████████████████████████████████| 18/18 [04:13<00:00, 14.10s/it]


保存完成: B题-示例数据\示例数据\附件2：财务报告\深交extract_result.xlsx


In [ ]:
folder_root = r"B题-示例数据\示例数据\附件2：财务报告"

all_core = []
all_balance = []
all_cash = []
all_income = []
all_other = []
